In [6]:
import pandas as pd
df = pd.read_csv("trimmed-data-1.csv")
print(df["Architectural Style"].isna().sum())

0


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd

df = pd.read_csv("trimmed-data-1.csv")

X = df.drop(columns=["Municipal", "Is Demolished"])
y = df["Municipal"]
X = pd.get_dummies(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

y_proba = model.predict_proba(X_test)[:, 1]
y_pred_adjusted = (y_proba >= 0.3).astype(bool)
print(classification_report(y_test, y_pred_adjusted))
print(df["Municipal"].value_counts())

              precision    recall  f1-score   support

       False       0.86      0.97      0.91       160
        True       0.75      0.32      0.45        37

    accuracy                           0.85       197
   macro avg       0.81      0.65      0.68       197
weighted avg       0.84      0.85      0.83       197

              precision    recall  f1-score   support

       False       0.87      0.88      0.87       160
        True       0.44      0.43      0.44        37

    accuracy                           0.79       197
   macro avg       0.66      0.65      0.66       197
weighted avg       0.79      0.79      0.79       197

Municipal
False    807
True     175
Name: count, dtype: int64


In [8]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

model = RandomForestClassifier(random_state=42)
model.fit(X_train_sm, y_train_sm)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.86      0.86      0.86       160
        True       0.38      0.38      0.38        37

    accuracy                           0.77       197
   macro avg       0.62      0.62      0.62       197
weighted avg       0.77      0.77      0.77       197



In [9]:
feature_importance = pd.Series(
    model.feature_importances_, 
    index=X_train_sm.columns
).sort_values(ascending=False).head(20)

print(feature_importance)

Year of Construction                                            0.121947
Ward                                                            0.054927
Original Use_Commerce                                           0.047670
Original Use_Residence                                          0.038924
Architectural Style_Queen Anne Revival                          0.031454
Resource Type_City Wide Historic Resource                       0.027551
Architectural Style_Edwardian Commercial                        0.026889
Community_BELTLINE                                              0.025857
Community_UPPER MOUNT ROYAL                                     0.025316
Architectural Style_Unknown                                     0.022327
Original Use_Religion, Ritual and Funeral                       0.021660
Development Era_1906 to 1913 (Pre WWI Boom, Age of Optimism)    0.021044
Resource Type_Community Historic Resource                       0.017676
Original Use_Education                             

In [10]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.86      0.86      0.86       160
        True       0.38      0.38      0.38        37

    accuracy                           0.77       197
   macro avg       0.62      0.62      0.62       197
weighted avg       0.77      0.77      0.77       197

